In [45]:
import pandas as pd 
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score


from fantasy_football.fpl_api.get_performance_data import get_most_recent_gw_points
from fantasy_football.model.training.utils import load_gw_data, create_correlation_matrix, add_historic_rolling_features
from fantasy_football.utils import create_optimal_team
from fantasy_football.fpl_api.get_live_data import LivePlayerData

In [46]:
historic_data = load_gw_data("2025-26")
historic_data.head()

,name,position,team,xP,assists,bonus,bps,clean_sheets,creativity,element,...,transfers_in,transfers_out,value,was_home,yellow_cards,clearances_blocks_interceptions,defensive_contribution,recoveries,tackles,GW
0,Reinildo Mandava,DEF,Sunderland,0.5,0,0,27,1,2.5,541,...,0,0,40,True,0,6,8,3,2,1
1,Lewis Dobbin,MID,Aston Villa,1.0,0,0,0,0,0.0,57,...,0,0,50,True,0,0,0,0,0,1
2,Ryan Christie,MID,Bournemouth,0.0,0,0,0,0,0.0,87,...,0,0,50,False,0,0,0,0,0,1
3,Zeki Amdouni,FWD,Burnley,0.0,0,0,0,0,0.0,216,...,0,0,50,False,0,0,0,0,0,1
4,Lucas Tolentino Coelho de Lima,MID,West Ham,2.6,0,0,11,0,14.2,612,...,0,0,60,False,0,0,6,5,1,1


In [47]:
target_variable = "total_points"

historic_data[target_variable].describe()

count    29757.000000
mean         1.156333
std          2.352553
min         -3.000000
25%          0.000000
50%          0.000000
75%          1.000000
max         24.000000
Name: total_points, dtype: float64

# Known ahead of time

In [48]:
known_features = [
    "name",
    "element",
    "position",
    "team",
    "GW",
    "fixture",
    "was_home",
]

# Features from lagged columns

In [49]:
historic_features = [
    'assists',
    'clean_sheets',
    'creativity',
    'goals_conceded',
    'goals_scored',
    'ict_index',
    'influence',
    'own_goals',
    'penalties_missed',
    'penalties_saved',
    'red_cards',
    'saves',
    'selected',
    'starts',
    'threat',
    'transfers_balance',
    'value',
    'yellow_cards',
    'clearances_blocks_interceptions',
    'defensive_contribution',
    'recoveries',
    'tackles',
]

for feature in historic_features:

    historic_data = add_historic_rolling_features(
        historic_data,
        feature_column=feature,
        windows = [1] #, 3, 6, 9, 12)
)

calculated_features = [col for col in historic_data.columns if "calc_" in col]


In [50]:
correlation_matrix = create_correlation_matrix(
    historic_data, 
    columns=calculated_features, 
    target_variable=target_variable)

# Filter to include where correlation to target is >0.1
correlation_matrix = correlation_matrix[ (0.1 < correlation_matrix) &  (correlation_matrix < 0.9) ]
correlation_matrix

calc_starts_mean_last_1_gw                             0.506949
calc_ict_index_mean_last_1_gw                          0.444393
calc_defensive_contribution_mean_last_1_gw             0.436339
calc_influence_mean_last_1_gw                          0.423308
calc_recoveries_mean_last_1_gw                         0.421038
calc_clearances_blocks_interceptions_mean_last_1_gw    0.358353
calc_goals_conceded_mean_last_1_gw                     0.358055
calc_creativity_mean_last_1_gw                         0.339697
calc_tackles_mean_last_1_gw                            0.328383
calc_threat_mean_last_1_gw                             0.322386
calc_selected_mean_last_1_gw                           0.298265
calc_value_mean_last_1_gw                              0.288221
calc_clean_sheets_mean_last_1_gw                       0.242051
calc_goals_scored_mean_last_1_gw                       0.173602
calc_yellow_cards_mean_last_1_gw                       0.163167
calc_assists_mean_last_1_gw             

# MVP model

In [51]:
# Split by complete gameweeks so future results never leak into training
validation_fraction = 0.2
gameweeks = sorted(historic_data["GW"].dropna().unique())
split_index = max(1, int(len(gameweeks) * (1 - validation_fraction)))
validation_gameweeks = gameweeks[split_index:]

model_features = known_features + correlation_matrix.index.tolist()

train_mask = ~historic_data["GW"].isin(validation_gameweeks)
valid_mask = historic_data["GW"].isin(validation_gameweeks)

X_train = historic_data.loc[train_mask, model_features].copy()
y_train = historic_data.loc[train_mask, target_variable].copy()
X_valid = historic_data.loc[valid_mask, model_features].copy()
y_valid = historic_data.loc[valid_mask, target_variable].copy()

print(f"Train: {len(X_train):,} rows through GW {gameweeks[split_index - 1]}")
print(f"Validation: {len(X_valid):,} rows from GW {validation_gameweeks[0]}")


Train: 23,175 rows through GW 30
Validation: 6,582 rows from GW 31


## Dummy model

In [52]:
from sklearn.dummy import DummyRegressor

categorical_columns = ["name", "element", "position", "team", "fixture", "was_home"]
for frame in (X_train, X_valid):
    frame[categorical_columns] = frame[categorical_columns].fillna("__MISSING__").astype(str)

models = {}
validation_predictions = {}

dummy_model = DummyRegressor(strategy="mean")
dummy_model.fit(X_train, y_train)
models["Dummy"] = dummy_model
validation_predictions["Dummy"] = dummy_model.predict(X_valid)

## Unweighted

In [53]:
def fit_catboost(sample_weight=None):
    fitted_model = CatBoostRegressor(
        iterations=163,
        learning_rate=0.03,
        depth=6,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
    )
    fitted_model.fit(
        X_train,
        y_train,
        cat_features=categorical_columns,
        sample_weight=sample_weight,
    )
    return fitted_model

unweighted_model = fit_catboost()
models["Unweighted"] = unweighted_model
validation_predictions["Unweighted"] = unweighted_model.predict(X_valid)

## Basic weighting
Weights higher if score was > 6 in GW

In [54]:
# Give high-scoring returns extra influence during training.
basic_weights = pd.Series(1.0, index=y_train.index)
basic_weights.loc[y_train > 6] = 4.0

basic_weighting_model = fit_catboost(sample_weight=basic_weights)
models["BASIC weighting"] = basic_weighting_model
validation_predictions["BASIC weighting"] = basic_weighting_model.predict(X_valid)

## Percentile weighting
Weights increase with the player's score percentile within their training gameweek.

In [55]:
train_gameweeks = historic_data.loc[y_train.index, "GW"]
score_percentiles = y_train.groupby(train_gameweeks).rank(pct=True, method="average")
percentile_weights = pd.cut(
    score_percentiles,
    bins=[0, 0.25, 0.50, 0.75, 0.90, 1.0],
    labels=[1.0, 1.5, 2.0, 3.0, 4.0],
    include_lowest=True,
).astype(float)

percentile_weighting_model = fit_catboost(sample_weight=percentile_weights)
models["Percentile weighting"] = percentile_weighting_model
validation_predictions["Percentile weighting"] = percentile_weighting_model.predict(X_valid)

# Comparison table

In [56]:
def evaluate_predictions(predictions, top_n=20):
    evaluation = pd.DataFrame({
        "GW": historic_data.loc[y_valid.index, "GW"].to_numpy(),
        "actual": y_valid.to_numpy(),
        "predicted": predictions,
    })

    top_end_by_gw = []
    for _, gameweek in evaluation.groupby("GW"):
        n = min(top_n, len(gameweek))
        predicted_top = gameweek.nlargest(n, "predicted")
        actual_top_indices = set(gameweek.nlargest(n, "actual").index)
        top_end_by_gw.append({
            "top_20_avg_actual_points": predicted_top["actual"].mean(),
            "top_20_hit_rate": predicted_top.index.isin(actual_top_indices).mean(),
            "top_20_oracle_regret": (
                gameweek.nlargest(n, "actual")["actual"].mean()
                - predicted_top["actual"].mean()
            ),
        })

    top_end = pd.DataFrame(top_end_by_gw).mean()
    return {
        "MAE": mean_absolute_error(evaluation["actual"], evaluation["predicted"]),
        "RMSE": root_mean_squared_error(evaluation["actual"], evaluation["predicted"]),
        "R2": r2_score(evaluation["actual"], evaluation["predicted"]),
        **top_end.to_dict(),
    }

comparison_table = pd.DataFrame.from_dict(
    {name: evaluate_predictions(preds) for name, preds in validation_predictions.items()},
    orient="index",
).rename_axis("model").reset_index()

comparison_table = comparison_table.sort_values(
    "top_20_avg_actual_points", ascending=False
).reset_index(drop=True)
# Use the strongest top-end model for the downstream forecast section.
model = models["Percentile weighting"]

comparison_table.round(3)

,model,MAE,RMSE,R2,top_20_avg_actual_points,top_20_hit_rate,top_20_oracle_regret
0,Percentile weighting,1.119,1.935,0.276,4.338,0.200,6.644
1,Unweighted,0.939,1.855,0.335,4.212,0.181,6.769
2,BASIC weighting,1.239,2.115,0.135,4.156,0.188,6.825
3,Dummy,1.495,2.276,-0.002,1.700,0.044,9.281


# For initial draft, maximise the average forecast points for the last 8 weeks of last season

In [57]:
X_valid['prediction'] = model.predict(X_valid)

forecast_data = X_valid.groupby('name')['prediction'].mean().reset_index()

live_data = LivePlayerData()

# get ["name", "position", "team", "value"]
forecast_data['player_id'] = forecast_data['name'].map(live_data.get_player_id)
forecast_data['position'] = forecast_data['name'].map(live_data.get_live_player_position)
forecast_data['team'] = forecast_data['name'].map(live_data.get_live_player_team)
forecast_data['value'] = forecast_data['name'].map(live_data.get_live_player_cost)



In [58]:
forecast_data = forecast_data.sort_values(by="prediction", ascending=False).dropna()

In [59]:
optimal_team = create_optimal_team(forecast_data,'value')

optimal_team['last_gw_points'] = optimal_team['player_id'].map(get_most_recent_gw_points)

In [60]:
optimal_team.head(20)

,name,prediction,player_id,position,team,value,last_gw_points
0,Erling Haaland,5.031387,411,FWD,Man City,155.0,2
1,Antoine Semenyo,4.867332,397,MID,Man City,85.0,2
2,Virgil van Dijk,4.850170,356,DEF,Liverpool,65.0,2
3,Viktor Gyökeres,4.364797,25,FWD,Arsenal,74.0,0
4,Cole Palmer,4.348044,154,MID,Chelsea,95.0,13
5,Nikola Milenković,4.246915,471,DEF,Nott'm Forest,55.0,2
6,João Pedro Junqueira de Jesus,4.154528,165,FWD,Chelsea,76.0,11
7,Dango Ouattara,4.098910,95,MID,Brentford,65.0,3
8,Phil Foden,3.808376,398,MID,Man City,70.0,1
9,Filip Jörgensen,0.376650,141,GKP,Chelsea,50.0,0


In [61]:
optimal_team.sum()

name              Erling HaalandAntoine SemenyoVirgil van DijkVi...
prediction                                                40.918683
player_id                 41139735625154471165953981413845159609340
position              FWDMIDDEFFWDMIDDEFFWDMIDMIDGKPDEFDEFGKPDEFMID
team              Man CityMan CityLiverpoolArsenalChelseaNott'm ...
value                                                        1000.0
last_gw_points                                                   36
dtype: object

# Fiddly variables to come back to

In [62]:
#     "opponent_team",  # TODO: map to opponent team name from each season 
# game time of day
# Game day of week
# Last season performance


# Double game weeks
# What variable do i optimise for?